# A Recurrent Network for Next-Word Prediction

## The Recurrent Cell

$$
h_t = \tanh\left( x_t W_x + h_{t-1} W_h + b_h \right), \qquad h_0 = \mathbf{0},
$$

$$
P(w_{t+1} = j \mid w_1, \ldots, w_t) = \text{softmax}\left( h_t W_o + b_o \right)_j .
$$

## The Data

In [ ]:
import re
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import nltk
import torch
import torch.nn as nn
from nltk.corpus import gutenberg

nltk.download("gutenberg", quiet=True)
tokens = re.findall(r"[a-z']+", gutenberg.raw("carroll-alice.txt").lower())

counts = Counter(tokens)
vocabulary = ["<unk>"] + [word for word, n in counts.most_common() if n >= 2]
word_to_id = {word: i for i, word in enumerate(vocabulary)}
VOCAB_SIZE = len(vocabulary)
UNK = word_to_id["<unk>"]

def encode(words):
    return [word_to_id.get(word, UNK) for word in words]

ids = np.array(encode(tokens))
n_train, n_val = int(0.8 * len(ids)), int(0.9 * len(ids))
segments = {"train": ids[:n_train], "val": ids[n_train:n_val], "test": ids[n_val:]}

print("tokens:", len(tokens), " vocabulary size:", VOCAB_SIZE)
print("share of <unk> tokens:", round((ids == UNK).mean(), 3))
print({name: len(segment) for name, segment in segments.items()})

### Training Examples

In [ ]:
CONTEXT = 8

def make_windows(segment, context=CONTEXT, start=CONTEXT):
    # contexts[i] = the `context` words before position t, targets[i] = the word at position t
    positions = np.arange(start, len(segment))
    contexts = np.stack([segment[t - context:t] for t in positions])
    return torch.tensor(contexts), torch.tensor(segment[positions])

X_train, y_train = make_windows(segments["train"])
X_val, y_val = make_windows(segments["val"])
X_test, y_test = make_windows(segments["test"])

print("contexts:", X_train.shape, X_val.shape, X_test.shape)
for i in (0, 1000):
    print(" ".join(vocabulary[j] for j in X_train[i]), "->", vocabulary[y_train[i]])

## The Networks

### A Recurrent Cell Written by Hand

In [ ]:
class ElmanRNN(nn.Module):
    # h_t = tanh(x_t W_x + h_{t-1} W_h + b_h); the logits are computed from the last state
    def __init__(self, vocab_size, embedding_size=32, hidden_size=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.W_x = nn.Parameter(torch.randn(embedding_size, hidden_size) / embedding_size ** 0.5)
        self.W_h = nn.Parameter(torch.randn(hidden_size, hidden_size) / hidden_size ** 0.5)
        self.b_h = nn.Parameter(torch.zeros(hidden_size))
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, word_ids):
        # word_ids: (n, L) integers -> logits (n, vocab_size)
        x = self.embedding(word_ids)                         # (n, L, e)
        h = torch.zeros(word_ids.shape[0], self.W_h.shape[0])
        for t in range(word_ids.shape[1]):
            h = torch.tanh(x[:, t] @ self.W_x + h @ self.W_h + self.b_h)
        return self.output(h)

### The Same Network with `nn.RNN` (and GRU, LSTM)

In [ ]:
class NextWordNet(nn.Module):
    def __init__(self, vocab_size, embedding_size=32, hidden_size=128, kind="rnn", dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size)
        layer = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}[kind]
        self.recurrent = layer(embedding_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, word_ids):
        # word_ids: (n, L); any length L >= 1 works
        states, _ = self.recurrent(self.embedding(word_ids))   # (n, L, hidden): the state after each word
        return self.output(self.dropout(states[:, -1]))

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

torch.manual_seed(0)
reference = NextWordNet(VOCAB_SIZE, kind="rnn")
by_hand = ElmanRNN(VOCAB_SIZE)

# copy the weights of the PyTorch model into the hand-written one (PyTorch stores W^T)
with torch.no_grad():
    by_hand.embedding.weight.copy_(reference.embedding.weight)
    by_hand.W_x.copy_(reference.recurrent.weight_ih_l0.T)
    by_hand.W_h.copy_(reference.recurrent.weight_hh_l0.T)
    by_hand.b_h.copy_(reference.recurrent.bias_ih_l0 + reference.recurrent.bias_hh_l0)
    by_hand.output.load_state_dict(reference.output.state_dict())

reference.eval()
with torch.no_grad():
    difference = (reference(X_train[:100]) - by_hand(X_train[:100])).abs().max().item()
print(f"largest difference between the two implementations: {difference:.2e}")

### Where Do the Parameters Come From?

## Training

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        return loss_fn(model(X), y).item()

def fit(model, X_tr, y_tr, X_va, y_va, n_epochs=25, learning_rate=3e-3, batch_size=128, seed=0):
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {"train": [], "val": []}
    best_loss, best_state = float("inf"), None
    for epoch in range(n_epochs):
        model.train()
        order = torch.randperm(len(X_tr))
        for start in range(0, len(order), batch_size):
            batch = order[start:start + batch_size]
            optimizer.zero_grad()
            loss_fn(model(X_tr[batch]), y_tr[batch]).backward()
            optimizer.step()
        history["train"].append(evaluate(model, X_tr, y_tr))
        history["val"].append(evaluate(model, X_va, y_va))
        if history["val"][-1] < best_loss:
            best_loss = history["val"][-1]
            best_state = {name: value.clone() for name, value in model.state_dict().items()}
            history["best_epoch"] = epoch
    model.load_state_dict(best_state)
    return model, history

torch.manual_seed(0)
rnn, hist_rnn = fit(NextWordNet(VOCAB_SIZE, kind="rnn"), X_train, y_train, X_val, y_val)

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(hist_rnn["train"], label="train")
ax.plot(hist_rnn["val"], label="validation")
ax.axvline(hist_rnn["best_epoch"], color="gray", linestyle=":", label="selected epoch")
ax.set(xlabel="epoch", ylabel="cross-entropy")
ax.legend()
plt.show()

### Baselines: Unigram and Bigram

In [ ]:
train_ids = segments["train"]
pair_counts = np.zeros((VOCAB_SIZE, VOCAB_SIZE))
np.add.at(pair_counts, (train_ids[:-1], train_ids[1:]), 1)

def smoothed_bigram(alpha):
    counts_ = pair_counts + alpha
    return counts_ / counts_.sum(axis=1, keepdims=True)

def baseline_cross_entropy(probs_for, X, y):
    return -np.mean(np.log(probs_for(X)[np.arange(len(y)), y.numpy()]))

# choose the smoothing constant on the validation set
alphas = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3]
val_ce = [baseline_cross_entropy(lambda X: smoothed_bigram(a)[X[:, -1].numpy()], X_val, y_val) for a in alphas]
ALPHA = alphas[int(np.argmin(val_ce))]
print("validation cross-entropy of the bigram model for alpha =", dict(zip(alphas, np.round(val_ce, 3))))
print("chosen alpha:", ALPHA)

bigram_probs = smoothed_bigram(ALPHA)
unigram_counts = np.bincount(train_ids, minlength=VOCAB_SIZE) + ALPHA
unigram_probs = unigram_counts / unigram_counts.sum()

ce_unigram = baseline_cross_entropy(lambda X: np.tile(unigram_probs, (len(X), 1)), X_test, y_test)
ce_bigram = baseline_cross_entropy(lambda X: bigram_probs[X[:, -1].numpy()], X_test, y_test)
ce_rnn = evaluate(rnn, X_test, y_test)

def top_k_accuracy(scores, y, k):
    top = np.argsort(-scores, axis=1)[:, :k]
    return (top == y.numpy()[:, None]).any(axis=1).mean()

with torch.no_grad():
    rnn.eval()
    rnn_scores = rnn(X_test).numpy()
bigram_scores = bigram_probs[X_test[:, -1].numpy()]

print(f"{'model':<10} {'test CE':>8} {'perplexity':>11} {'top-1':>7} {'top-5':>7}")
for name, ce, scores in [("unigram", ce_unigram, np.tile(unigram_probs, (len(X_test), 1))),
                         ("bigram", ce_bigram, bigram_scores),
                         ("RNN", ce_rnn, rnn_scores)]:
    print(f"{name:<10} {ce:>8.3f} {np.exp(ce):>11.1f} {top_k_accuracy(scores, y_test, 1):>7.3f} {top_k_accuracy(scores, y_test, 5):>7.3f}")

### Gated Variants

In [ ]:
results = {"rnn": (rnn, hist_rnn)}
for kind in ["gru", "lstm"]:
    torch.manual_seed(0)
    results[kind] = fit(NextWordNet(VOCAB_SIZE, kind=kind), X_train, y_train, X_val, y_val)

print(f"{'layer':<6} {'parameters':>11} {'best epoch':>11} {'val perplexity':>15} {'test perplexity':>16}")
for kind, (model, history) in results.items():
    print(f"{kind:<6} {count_parameters(model):>11} {history['best_epoch'] + 1:>11} "
          f"{np.exp(min(history['val'])):>15.1f} {np.exp(evaluate(model, X_test, y_test)):>16.1f}")

## What Can We Do with a Next-Word Model?

### 1. Next-Word Suggestions

In [ ]:
def next_word_probs(model, words):
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor([encode(words)]))[0]
    logits[UNK] = float("-inf")
    return torch.softmax(logits, dim=0)

def suggest(model, prompt, k=5):
    probs = next_word_probs(model, prompt.lower().split())
    top = torch.topk(probs, k)
    return ", ".join(f"{vocabulary[i]} ({p:.2f})" for p, i in zip(top.values, top.indices))

for prompt in ["alice was beginning to get very", "she went on and", "the queen of", "off with her", "said the mock"]:
    print(f"{prompt!r:<40} -> {suggest(rnn, prompt)}")

### 2. The Context Matters

### 3. Text Generation

In [ ]:
def generate(model, prompt, n_words=25, temperature=1.0, seed=0, context=CONTEXT):
    generator = torch.Generator().manual_seed(seed)
    words = prompt.lower().split()
    model.eval()
    for _ in range(n_words):
        with torch.no_grad():
            logits = model(torch.tensor([encode(words[-context:])]))[0]
        logits[UNK] = float("-inf")
        if temperature == 0:
            next_id = logits.argmax().item()
        else:
            next_id = torch.multinomial(torch.softmax(logits / temperature, dim=0), 1, generator=generator).item()
        words.append(vocabulary[next_id])
    return " ".join(words)

def generate_bigram(prompt, n_words=25, seed=0):
    generator = np.random.default_rng(seed)
    words = prompt.lower().split()
    for _ in range(n_words):
        probs = bigram_probs[word_to_id.get(words[-1], UNK)].copy()
        probs[UNK] = 0
        words.append(vocabulary[generator.choice(VOCAB_SIZE, p=probs / probs.sum())])
    return " ".join(words)

prompt = "alice was beginning to get very"
print("bigram           :", generate_bigram(prompt))
print("RNN, greedy      :", generate(rnn, prompt, temperature=0))
for temperature in [0.5, 1.0, 1.5]:
    print(f"RNN, tau = {temperature:<4}  :", generate(rnn, prompt, temperature=temperature))

### 4. Scoring Sentences

In [ ]:
def sentence_perplexity(model, words):
    # perplexity per word; the first word has no context and is not scored
    model.eval()
    ids_ = torch.tensor([encode(words)])
    log_probs = []
    for t in range(1, len(words)):
        with torch.no_grad():
            logits = model(ids_[:, :t])[0]
        log_probs.append(torch.log_softmax(logits, dim=0)[ids_[0, t]].item())
    return float(np.exp(-np.mean(log_probs)))

test_words = [vocabulary[i] for i in segments["test"]]
rng = np.random.default_rng(0)
original, shuffled = [], []
for start in range(0, len(test_words) - 12, 12):
    excerpt = test_words[start:start + 12]
    if "<unk>" in excerpt:
        continue
    original.append(sentence_perplexity(rnn, excerpt))
    shuffled.append(sentence_perplexity(rnn, list(rng.permutation(excerpt))))

original, shuffled = np.array(original), np.array(shuffled)
print("excerpts:", len(original))
print(f"median perplexity per word: original {np.median(original):.0f}, shuffled {np.median(shuffled):.0f}")
print(f"original text has the lower perplexity in {(original < shuffled).mean():.0%} of the excerpts")

### 5. What Did the Embeddings Learn?

## Limitations

## Summary

## Exercises